# simpson_paradox

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/simpson_paradox.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们模拟 1000 个个体，并生成性别虚拟变量。

In [2]:
%%stata
clear
set obs 1000
gen sex=runiform()<0.5


. clear

. set obs 1000
Number of observations (_N) was 0, now 1,000.

. gen sex=runiform()<0.5

. 


我们让锻炼习惯（exer）的分布依赖于性别：男性锻炼概率 0.8，女性仅 0.3。这样性别同时影响锻炼和结果变量，成为一个混杂因素。

In [3]:
%%stata
gen     exer=runiform()<0.8 if sex==1
replace exer=runiform()<0.3 if sex==0


. gen     exer=runiform()<0.8 if sex==1
(495 missing values generated)

. replace exer=runiform()<0.3 if sex==0
(495 real changes made)

. 


我们生成结果变量 y：其真实模型中锻炼的效应为 3，性别效应为 -10，并附加标准正态噪声。锻炼的真实效应是正的，但锻炼与性别正相关、性别效应为负，混杂会让全样本估计失真。

In [4]:
%%stata
gen y=80-10*sex+3*exer+rnormal()

我们先在合起来的全样本中回归 y 对 exer。由于遗漏了性别这一混杂因素，exer 的系数吸收了性别差异，甚至可能呈现虚假的负相关。

In [5]:
%%stata
reg y exer
outreg2 using simpson_paradox.tex, replace ctitle(全样本)


. reg y exer

      Source |       SS           df       MS      Number of obs   =     1,000
-------------+----------------------------------   F(1, 998)       =     64.05
       Model |  1222.64837         1  1222.64837   Prob > F        =    0.0000
    Residual |  19050.7646       998  19.0889425   R-squared       =    0.0603
-------------+----------------------------------   Adj R-squared   =    0.0594
       Total |  20273.4129       999  20.2937066   Root MSE        =    4.3691

------------------------------------------------------------------------------
           y | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        exer |  -2.216301   .2769293    -8.00   0.000    -2.759731   -1.672871
       _cons |   77.73535   .2021773   384.49   0.000     77.33861    78.13209
------------------------------------------------------------------------------

. outreg2 using simpson_paradox.tex

接着我们分性别分别回归。在每组内部，锻炼与性别不再相关，系数应接近真实效应 3。

In [6]:
%%stata
reg y exer if sex==1
outreg2 using simpson_paradox.tex, append ctitle(男性)
reg y exer if sex==0
outreg2 using simpson_paradox.tex, append ctitle(女性)


. reg y exer if sex==1

      Source |       SS           df       MS      Number of obs   =       505
-------------+----------------------------------   F(1, 503)       =    684.50
       Model |  671.869688         1  671.869688   Prob > F        =    0.0000
    Residual |  493.715495       503  .981541739   R-squared       =    0.5764
-------------+----------------------------------   Adj R-squared   =    0.5756
       Total |  1165.58518       504  2.31266901   Root MSE        =    .99073

------------------------------------------------------------------------------
           y | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        exer |   2.822635   .1078864    26.16   0.000     2.610672    3.034599
       _cons |   70.15965   .0957773   732.53   0.000     69.97148    70.34783
------------------------------------------------------------------------------

. outreg2 using simpson_p

最后我们在全样本回归中加入性别控制变量。控制混杂因素后，exer 的系数同样恢复到 3 附近。这正是辛普森悖论：合并数据中的关系方向可以与分组数据相反，而加入适当的控制变量即可消除混杂、恢复真实效应。

In [7]:
%%stata
reg y exer sex
outreg2 using simpson_paradox.tex, append ctitle(全样本)


. reg y exer sex

      Source |       SS           df       MS      Number of obs   =     1,000
-------------+----------------------------------   F(2, 997)       =   9873.46
       Model |  19299.0265         2  9649.51327   Prob > F        =    0.0000
    Residual |  974.386396       997  .977318351   R-squared       =    0.9519
-------------+----------------------------------   Adj R-squared   =    0.9518
       Total |  20273.4129       999  20.2937066   Root MSE        =    .98859

------------------------------------------------------------------------------
           y | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        exer |   2.923843   .0731769    39.96   0.000     2.780244    3.067441
         sex |  -9.930829    .073021  -136.00   0.000    -10.07412   -9.787536
       _cons |   80.01072   .0487101  1642.59   0.000     79.91513    80.10631
--------------------------------